In [1]:
import pandas as pd
from pathlib import Path
import json
import time
from datetime import datetime

def create_month_parquets(base_dir: Path, output_dir: Path, months: range = range(1, 15)):
    """
    Create parquet files for each month with job IDs and folder names.
    Memory-efficient: processes one month at a time, writes in chunks.
    
    Args:
        base_dir: Base directory containing month folders
        output_dir: Where to save parquet files
        months: Range of months to process (default 1-14)
    """
    output_dir.mkdir(parents=True, exist_ok=True)
    
    print(f"Started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    start_total = time.time()
    
    for month in months:
        folder_name = f"adzuna_month{month:02d}_npz"
        month_dir = base_dir / folder_name
        
        if not month_dir.exists():
            print(f"⚠️  Skipping {folder_name} - directory not found")
            continue
        
        start_month = time.time()
        output_file = output_dir / f"{folder_name}.parquet"
        
        # Process in chunks - write directly without storing all in memory
        first_chunk = True
        total_ids = 0
        
        for jsonl_file in month_dir.glob("*.jsonl"):
            chunk_ids = []
            
            with open(jsonl_file, 'r') as f:
                for line in f:
                    if line.strip():
                        try:
                            record = json.loads(line)
                            if 'id' in record:
                                chunk_ids.append(record['id'])
                        except:
                            continue
            
            if chunk_ids:
                df_chunk = pd.DataFrame({
                    'id': chunk_ids,
                    'folder_name': folder_name
                })
                
                # Write chunk to parquet (append mode)
                df_chunk.to_parquet(
                    output_file,
                    index=False,
                    engine='fastparquet',
                    append=not first_chunk
                )
                
                total_ids += len(chunk_ids)
                first_chunk = False
                del df_chunk, chunk_ids  # explicit cleanup
        
        duration_month = time.time() - start_month
        print(f"✓ {folder_name}: {total_ids} IDs -> {output_file.name} ({duration_month:.2f}s)")
    
    duration_total = time.time() - start_total
    print(f"\nCompleted at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"Total duration: {duration_total:.2f}s ({duration_total/60:.2f}m)")

# Usage:
base_dir = Path("/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_1_LLM_summary")
output_dir = Path("/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_1_LLM_summary/month_ids")

create_month_parquets(base_dir, output_dir)

Started at: 2026-02-16 18:37:32

✓ adzuna_month01_npz: 2658928 IDs -> adzuna_month01_npz.parquet (32.86s)
✓ adzuna_month02_npz: 2230469 IDs -> adzuna_month02_npz.parquet (26.08s)
✓ adzuna_month03_npz: 2304918 IDs -> adzuna_month03_npz.parquet (35.08s)
✓ adzuna_month04_npz: 2332783 IDs -> adzuna_month04_npz.parquet (38.62s)
✓ adzuna_month05_npz: 2547671 IDs -> adzuna_month05_npz.parquet (29.52s)
✓ adzuna_month06_npz: 1993750 IDs -> adzuna_month06_npz.parquet (31.88s)
✓ adzuna_month07_npz: 2184408 IDs -> adzuna_month07_npz.parquet (31.33s)
✓ adzuna_month08_npz: 1914855 IDs -> adzuna_month08_npz.parquet (36.29s)
✓ adzuna_month09_npz: 1861164 IDs -> adzuna_month09_npz.parquet (26.23s)
✓ adzuna_month10_npz: 2180380 IDs -> adzuna_month10_npz.parquet (30.26s)
✓ adzuna_month11_npz: 2041412 IDs -> adzuna_month11_npz.parquet (27.49s)
✓ adzuna_month12_npz: 1740199 IDs -> adzuna_month12_npz.parquet (24.58s)
✓ adzuna_month13_npz: 2551779 IDs -> adzuna_month13_npz.parquet (22.34s)
✓ adzuna_month14_n

In [2]:
import pandas as pd
from pathlib import Path
import time
from datetime import datetime

print(f"Started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
start_time = time.time()

input_dir = Path("/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_1_LLM_summary/month_ids")
output_file = Path("/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_1_LLM_summary/month_ids/jsonl_all_months_deduplicated.parquet")

# Collect all parquet files
parquet_files = []
for month in range(1, 15):
    pq_file = input_dir / f"adzuna_month{month:02d}_npz.parquet"
    if pq_file.exists():
        parquet_files.append(pq_file)
        print(f"Found: {pq_file.name}")

print(f"\nConcatenating {len(parquet_files)} files...")

# Read all files and concatenate
dfs = []
for pq_file in parquet_files:
    df = pd.read_parquet(pq_file)
    dfs.append(df)
    print(f"  Loaded {pq_file.name}: {len(df):,} rows")

df_combined = pd.concat(dfs, ignore_index=True)
del dfs

print(f"\nTotal rows before sorting: {len(df_combined):,}")

# Sort by folder_name
df_combined = df_combined.sort_values('folder_name').reset_index(drop=True)
print(f"Sorted by folder_name")

# Drop duplicates by id (keeps first = earliest month)
rows_before = len(df_combined)
df_deduplicated = df_combined.drop_duplicates(subset='id', keep='first').reset_index(drop=True)
del df_combined

duplicates_removed = rows_before - len(df_deduplicated)
print(f"\nDuplicates removed: {duplicates_removed:,}")
print(f"Final rows: {len(df_deduplicated):,}")
duration = time.time() - start_time
print(f"\nSaved to: {output_file}")
print(f"Completed at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Duration: {duration:.2f}s ({duration/60:.2f}m)")


Started at: 2026-02-16 18:51:52

Found: adzuna_month01_npz.parquet
Found: adzuna_month02_npz.parquet
Found: adzuna_month03_npz.parquet
Found: adzuna_month04_npz.parquet
Found: adzuna_month05_npz.parquet
Found: adzuna_month06_npz.parquet
Found: adzuna_month07_npz.parquet
Found: adzuna_month08_npz.parquet
Found: adzuna_month09_npz.parquet
Found: adzuna_month10_npz.parquet
Found: adzuna_month11_npz.parquet
Found: adzuna_month12_npz.parquet
Found: adzuna_month13_npz.parquet
Found: adzuna_month14_npz.parquet

Concatenating 14 files...
  Loaded adzuna_month01_npz.parquet: 2,658,928 rows
  Loaded adzuna_month02_npz.parquet: 2,230,469 rows
  Loaded adzuna_month03_npz.parquet: 2,304,918 rows
  Loaded adzuna_month04_npz.parquet: 2,332,783 rows
  Loaded adzuna_month05_npz.parquet: 2,547,671 rows
  Loaded adzuna_month06_npz.parquet: 1,993,750 rows
  Loaded adzuna_month07_npz.parquet: 2,184,408 rows
  Loaded adzuna_month08_npz.parquet: 1,914,855 rows
  Loaded adzuna_month09_npz.parquet: 1,861,164 r

In [7]:
for i in df_deduplicated.folder_name.unique():
    print(i)
    print(len(df_deduplicated[df_deduplicated.folder_name == i]))


adzuna_month01_npz
2658928
adzuna_month02_npz
1539725
adzuna_month03_npz
1685375
adzuna_month04_npz
1501581
adzuna_month05_npz
1846101
adzuna_month06_npz
1288079
adzuna_month07_npz
1484989
adzuna_month08_npz
1297484
adzuna_month09_npz
1269625
adzuna_month10_npz
1486950
adzuna_month11_npz
1419868
adzuna_month12_npz
982586
adzuna_month13_npz
1226948
adzuna_month14_npz
8474


In [9]:
import numpy as np
from pathlib import Path

npz_path = Path("/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/job_ads_texts_by_month_npz/adzuna_month01.npz")

print(f"File exists: {npz_path.exists()}")
print(f"File size: {npz_path.stat().st_size / 1024**2:.2f} MB\n")

# Load and inspect
data = np.load(npz_path, allow_pickle=True)

print("Keys in NPZ file:")
print(data.files)
print()

# Show info for each array
for key in data.files:
    arr = data[key]
    print(f"\n{key}:")
    print(f"  dtype: {arr.dtype}")
    print(f"  shape: {arr.shape}")
    
    # Show first few values
    if len(arr) > 0:
        print(f"  first 3 values:")
        for i in range(min(3, len(arr))):
            val = arr[i]
            if isinstance(val, (str, np.str_)):
                print(f"    [{i}]: {val[:100]}..." if len(str(val)) > 100 else f"    [{i}]: {val}")
            else:
                print(f"    [{i}]: {val}")

data.close()

File exists: True
File size: 1122.25 MB

Keys in NPZ file:
['job_ids', 'role_text', 'taskskill_text', 'domains']


job_ids:
  dtype: object
  shape: (2657586,)
  first 3 values:
    [0]: 2746717438
    [1]: 2791837895
    [2]: 2746717653

role_text:
  dtype: object
  shape: (2657586,)
  first 3 values:
    [0]: Site Bench Officer for a security role at Battersea, requiring SIA licensure and excellent customer ...
    [1]: Junior Project Manager in the Engineering/Manufacturing sector
    [2]: Container Administrator

taskskill_text:
  dtype: object
  shape: (2657586,)
  first 3 values:
    [0]: Engage with visitors, guests, and staff of the estate and its accompanying buildings Exhibit excelle...
    [1]: Customer/stakeholder management Budget and schedule management Working with technical and commercial...
    [2]: Support the Container team Assist in administrative practices Provide excellent service Join a full-...

domains:
  dtype: object
  shape: (2657586,)
  first 3 values:
    